# GSHA — Graph-Spectral Hyperbolic Attention
## Interactive Architecture & Data Explorer
#### Wheat Futures Direction Prediction

---

**Six interactive building-block visualizations:**

| # | Section | Key Insight |
|---|---------|-------------|
| 1 | 📰 News Corpus Overview | Sentiment signal across 15 years of wheat news |
| 2 | 🕸️ Dynamic News Graph | Semantic + temporal edges — **hover a node to see headlines** |
| 3 | 🫧 Poincaré Disk | How hyperbolic geometry encodes news hierarchy |
| 4 | 📡 Chebyshev Spectral Conv | Polynomial graph filters in the frequency domain |
| 5 | 🎯 Hybrid Attention | Causal Euclidean × hyperbolic attention weights |
| 6 | 🔀 Architecture Flow | End-to-end GSHA information flow (Sankey) |

In [ ]:
import subprocess, sys
for _pkg in ["plotly", "networkx"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import networkx as nx
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Dark presentation palette ─────────────────────────────────────────────────
GOLD  = "#D4A017"; DBLUE = "#1E3A5F"; RED  = "#E74C3C"; GREEN = "#2ECC71"
CYAN  = "#00BCD4"; PURP  = "#9B59B6"; GRAY = "#95A5A6"
BG    = "#0D1117"; PBGC  = "#161B22"; TXT  = "#E6EDF3"; GRID  = "#30363D"

_HL = dict(bgcolor="#1C2333", bordercolor=GOLD,
           font=dict(color=TXT, size=12), align="left", namelength=-1)

def _dark(**kw):
    d = dict(paper_bgcolor=PBGC, plot_bgcolor=BG,
             font=dict(color=TXT, family="Inter, Arial, sans-serif"),
             margin=dict(l=60, r=40, t=90, b=60),
             hoverlabel=_HL)
    d.update(kw)
    return d

_SYM = dict(  # sentiment colorscale
    colorscale=[[0.0,"#E74C3C"],[0.45,"#95A5A6"],[0.55,"#95A5A6"],[1.0,"#2ECC71"]],
    cmin=-0.4, cmax=0.4,
    colorbar=dict(
        title=dict(text="Sentiment", font=dict(color=TXT)),
        tickfont=dict(color=TXT), thickness=12, len=0.55,
        tickvals=[-0.35, 0, 0.35], ticktext=["Bearish","Neutral","Bullish"],
    ),
)

print("✓  Libraries ready")

---
## 1 · News Corpus Overview

FinBERT sentiment scores and article volume since 2010. The signal is sparse but meaningful — news shocks align with major wheat-market events (Black Sea disruptions, harvest revisions, macro risk-off).

In [ ]:
BASE = "data"

# ── Price ─────────────────────────────────────────────────────────────────────
price_df = pd.read_csv(f"{BASE}/wheat_prices.csv")
price_df["Date"] = pd.to_datetime(price_df["Date"])
price_df = price_df.sort_values("Date").set_index("Date")
for _c in ["Price", "Open", "High", "Low"]:
    price_df[_c] = price_df[_c].replace({",": ""}, regex=True).astype(float)

# ── News ──────────────────────────────────────────────────────────────────────
news_df = pd.read_csv(f"{BASE}/wheat_news.csv")
news_df["date_day"] = pd.to_datetime(news_df["date"]).dt.strftime("%Y-%m-%d")
date_headlines = defaultdict(list)
date_sources   = defaultdict(list)
for _, row in news_df.iterrows():
    date_headlines[row["date_day"]].append(str(row["title"]))
    date_sources[row["date_day"]].append(str(row.get("source", "")))

# ── Sentiment ─────────────────────────────────────────────────────────────────
sent_df   = pd.read_csv(f"{BASE}/daily_news_sentiment.csv")
sent_df["date"] = pd.to_datetime(sent_df["date"])
sent_dict  = dict(zip(sent_df["date"].dt.strftime("%Y-%m-%d"), sent_df["sentiment_score"]))
count_dict = dict(zip(sent_df["date"].dt.strftime("%Y-%m-%d"), sent_df["article_count"]))

# ── Embeddings ────────────────────────────────────────────────────────────────
embeddings = torch.load(f"{BASE}/daily_news_embeddings.pt", weights_only=False)
emb_dates  = sorted(embeddings.keys())
news_dates = sorted(set(emb_dates) & set(date_headlines.keys()))

print(f"  Wheat price rows : {len(price_df):,}")
print(f"  News articles    : {len(news_df):,}")
print(f"  News days (w/ emb): {len(news_dates):,}  [{news_dates[0]} → {news_dates[-1]}]")
print(f"  Embedding dims   : {embeddings[emb_dates[0]].shape[0]}-D")

In [ ]:
sp = sent_df.copy().sort_values("date")
pos = sp[sp["sentiment_score"] >= 0]
neg = sp[sp["sentiment_score"] <  0]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.68, 0.32], vertical_spacing=0.06,
    subplot_titles=["Daily FinBERT Sentiment Score", "Articles per Day"],
)

# Positive fill
fig.add_trace(go.Scatter(
    x=pos["date"], y=pos["sentiment_score"],
    fill="tozeroy", mode="lines",
    line=dict(color=GREEN, width=1.2),
    fillcolor="rgba(46,204,113,0.22)", name="Positive",
    hovertemplate="%{x|%b %d %Y}<br>Sentiment: <b>%{y:.3f}</b><extra>Positive</extra>",
), row=1, col=1)

# Negative fill
fig.add_trace(go.Scatter(
    x=neg["date"], y=neg["sentiment_score"],
    fill="tozeroy", mode="lines",
    line=dict(color=RED, width=1.2),
    fillcolor="rgba(231,76,60,0.22)", name="Negative",
    hovertemplate="%{x|%b %d %Y}<br>Sentiment: <b>%{y:.3f}</b><extra>Negative</extra>",
), row=1, col=1)

# Zero line
fig.add_hline(y=0, line_dash="dash", line_color=GRAY, line_width=0.8, row=1, col=1)

# 30-day rolling average
roll = sp.set_index("date")["sentiment_score"].rolling("30D").mean().reset_index()
fig.add_trace(go.Scatter(
    x=roll["date"], y=roll["sentiment_score"],
    mode="lines", line=dict(color=GOLD, width=2, dash="solid"),
    name="30-day avg",
    hovertemplate="%{x|%b %d %Y}<br>30-day avg: <b>%{y:.3f}</b><extra></extra>",
), row=1, col=1)

# Article volume
fig.add_trace(go.Bar(
    x=sp["date"], y=sp["article_count"],
    marker_color=GOLD, opacity=0.75, name="Articles",
    hovertemplate="%{x|%b %d %Y}<br>Articles: <b>%{y}</b><extra></extra>",
), row=2, col=1)

fig.update_layout(
    height=540, showlegend=True,
    legend=dict(bgcolor="rgba(0,0,0,0)", x=0.01, y=0.98,
                font=dict(color=TXT)),
    **_dark(title=dict(
        text="<b>Wheat Futures News Corpus</b>  ·  FinBERT Sentiment & Coverage",
        font=dict(color=TXT, size=18))),
)
fig.update_xaxes(showgrid=False, tickfont=dict(color=GRAY))
fig.update_yaxes(gridcolor=GRID, zerolinecolor=GRID, tickfont=dict(color=GRAY))
fig.show()

---
## 2 · Dynamic News Graph Construction

`DynamicNewsGraphBuilder` constructs an adjacency matrix over the news-day window as a hybrid of:

$$A_{ij} = \alpha \cdot \underbrace{\cos\!\bigl(\mathbf{e}_i,\,\mathbf{e}_j\bigr)}_{\text{semantic}} + \beta \cdot \underbrace{e^{-\gamma|i-j|}}_{\text{temporal prior}}$$

where $\alpha, \beta, \gamma$ are **learnable parameters** (initialised at 0.7 / 0.3 / 0.1).
Edges are sparsified to the top-$k$ neighbours per node (*k* = 4).

> **Hover over any node** to read the news headlines for that day.

In [ ]:
# ── Build graph on 60 most-recent news days ───────────────────────────────────
VIZ_N = 60
viz_dates = news_dates[-VIZ_N:]

# Embedding matrix
E = np.stack([embeddings[d].numpy() for d in viz_dates])       # (N, 16)
cos_sim   = cosine_similarity(E)                                # semantic

# Temporal decay prior
idx  = np.arange(VIZ_N)
temp = np.exp(-0.1 * np.abs(idx[:, None] - idx[None, :]))      # temporal

# Combined adjacency (typical trained values)
adj = 0.70 * cos_sim + 0.30 * temp
np.fill_diagonal(adj, 0)

# k-NN sparsification (symmetric)
K = 4
adj_knn = np.zeros_like(adj)
for i in range(VIZ_N):
    top = np.argsort(adj[i])[-K:]
    adj_knn[i, top] = adj[i, top]
adj_knn = np.maximum(adj_knn, adj_knn.T)

# NetworkX spring layout
G   = nx.from_numpy_array(adj_knn)
pos = nx.spring_layout(G, seed=42, k=2.8 / np.sqrt(VIZ_N), iterations=60)

print(f"Graph → {G.number_of_nodes()} nodes | {G.number_of_edges()} edges")
print(f"Mean degree: {np.mean([d for _, d in G.degree()]):.1f}")
print(f"Date range : {viz_dates[0]} → {viz_dates[-1]}")

In [ ]:
# ── Build Plotly traces ───────────────────────────────────────────────────────

# --- EDGES (grouped by weight quartile for opacity variation) ----------------
edge_groups = {
    "strong": ([], [], "rgba(100,160,255,0.70)", 2.0),
    "medium": ([], [], "rgba(100,160,255,0.40)", 1.2),
    "weak"  : ([], [], "rgba(100,160,255,0.18)", 0.6),
}
wts = nx.get_edge_attributes(G, "weight")
all_wts = list(wts.values())
q33, q66 = np.percentile(all_wts, 33), np.percentile(all_wts, 66)

for (u, v), w in wts.items():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    seg = [x0, x1, None], [y0, y1, None]
    key = "strong" if w > q66 else ("medium" if w > q33 else "weak")
    edge_groups[key][0].extend(seg[0])
    edge_groups[key][1].extend(seg[1])

edge_traces = [
    go.Scatter(x=ex, y=ey, mode="lines", hoverinfo="none", showlegend=False,
               line=dict(width=lw, color=col), name=lbl)
    for lbl, (ex, ey, col, lw) in edge_groups.items()
    if ex  # skip empty groups
]

# --- NODES ------------------------------------------------------------------
nx_vals = [pos[i][0] for i in range(VIZ_N)]
ny_vals = [pos[i][1] for i in range(VIZ_N)]
sents   = [sent_dict.get(d, 0.0) for d in viz_dates]
cnts    = [count_dict.get(d, 1)  for d in viz_dates]

hover_texts = []
for i, d in enumerate(viz_dates):
    heads = date_headlines.get(d, [])
    srcs  = date_sources.get(d, [])
    lines = "<br>".join(
        f'<span style="color:{GOLD}">▸</span> '
        f'<i style="color:{GRAY}">{s}</i>: '
        f'{h[:95]}{"…" if len(h) > 95 else ""}'
        for s, h in zip(srcs[:6], heads[:6])
    )
    sc   = sent_dict.get(d, 0.0)
    lbl  = "Bullish" if sc > 0.05 else ("Bearish" if sc < -0.05 else "Neutral")
    col  = GREEN if sc > 0.05 else (RED if sc < -0.05 else GRAY)
    hover_texts.append(
        f'<b style="font-size:14px">📅 {d}</b><br>'
        f'Sentiment: <b style="color:{col}">{sc:+.3f}  ({lbl})</b><br>'
        f'Articles: <b>{count_dict.get(d, 1)}</b>'
        + (f'<br><br><b>Headlines:</b><br>{lines}' if lines else "")
    )

node_trace = go.Scatter(
    x=nx_vals, y=ny_vals, mode="markers",
    hoverinfo="text", hovertext=hover_texts,
    marker=dict(
        size=[max(9, min(22, 9 + c * 3.5)) for c in cnts],
        color=sents,
        **_SYM,
        line=dict(width=1.8, color="rgba(255,255,255,0.35)"),
    ),
    name="News day",
)

# --- FIGURE -----------------------------------------------------------------
fig = go.Figure(data=[*edge_traces, node_trace])
fig.update_layout(
    title=dict(
        text=(
            "<b>GSHA — Dynamic News Graph</b>"
            "<br><sup>Node color = FinBERT sentiment · Size = article count · "
            "Edge opacity = similarity strength · <b>Hover a node to read headlines</b></sup>"
        ),
        font=dict(color=TXT, size=18),
    ),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=680,
    hovermode="closest",
    showlegend=False,
    **_dark(margin=dict(l=10, r=10, t=100, b=10)),
)
fig.show()

---
## 3 · Hyperbolic Embedding Space — Poincaré Disk

Standard Euclidean space is *flat* — equal distances everywhere.
The **Poincaré disk** $\mathbb{D}^2_c$ is *negatively curved*: points near the boundary are exponentially farther apart than they appear, naturally encoding **tree-like hierarchy**.

Each news-day embedding $\mathbf{e} \in \mathbb{R}^{16}$ is:
1. PCA-compressed to $\mathbb{R}^{2}$
2. Mapped via the **exponential map at the origin**: $\exp_{\mathbf{0}}(\mathbf{v}) = \tanh(\sqrt{c}\,\|\mathbf{v}\|) \cdot \frac{\mathbf{v}}{\sqrt{c}\,\|\mathbf{v}\|}$

Points *close to the boundary* = narrow/specific events.
Points *near the centre* = broad macro themes.

In [ ]:
# ── Poincaré disk projection ──────────────────────────────────────────────────
DISK_N = min(120, len(news_dates))
disk_dates = news_dates[-DISK_N:]
E_all  = np.stack([embeddings[d].numpy() for d in disk_dates])

pca2  = PCA(n_components=2, random_state=42)
xy    = pca2.fit_transform(E_all)                    # (N, 2)

# Normalise so largest norm → 0.95 inside disk
r_max  = np.linalg.norm(xy, axis=1).max() * 1.05
xy_n   = xy / r_max * 0.95                           # inside unit disk

# Exponential map  v → tanh(||v||) * v/||v||
norms  = np.linalg.norm(xy_n, axis=1, keepdims=True) + 1e-8
xy_d   = np.tanh(norms) * xy_n / norms               # Poincaré coords

sents_d = [sent_dict.get(d, 0.0) for d in disk_dates]
hovers_d = []
for d in disk_dates:
    heads = date_headlines.get(d, [])
    h_str = "<br>".join(f"▸ {h[:85]}" for h in heads[:4])
    sc    = sent_dict.get(d, 0.0)
    hovers_d.append(
        f"<b>📅 {d}</b><br>Sentiment: <b>{sc:+.3f}</b>"
        + (f"<br><br>{h_str}" if h_str else "")
    )

# ── Figure ───────────────────────────────────────────────────────────────────
fig = go.Figure()

# Boundary & concentric rings
theta = np.linspace(0, 2 * np.pi, 300)
fig.add_trace(go.Scatter(
    x=np.cos(theta), y=np.sin(theta), mode="lines",
    line=dict(color=GOLD, width=2, dash="dash"), name="∂D² (boundary)",
    hoverinfo="none",
))
for r in [0.33, 0.66]:
    fig.add_trace(go.Scatter(
        x=r * np.cos(theta), y=r * np.sin(theta), mode="lines",
        line=dict(color=GRID, width=0.6), showlegend=False, hoverinfo="none",
    ))

# Nodes
fig.add_trace(go.Scatter(
    x=xy_d[:, 0], y=xy_d[:, 1], mode="markers",
    hoverinfo="text", hovertext=hovers_d, name="News day",
    marker=dict(size=9, color=sents_d, **_SYM,
                line=dict(width=1.2, color="rgba(255,255,255,0.3)")),
))

# Annotations
for txt, x, y, ax, ay in [
    ("Macro themes\n(central)", 0.02, 0.04, 0, 0),
    ("Specific events\n(boundary)", 0.70, 0.54, 35, -30),
]:
    fig.add_annotation(x=x, y=y, text=txt.replace("\n", "<br>"),
                       showarrow=(txt != "Macro themes\n(central)"),
                       arrowcolor=GOLD, arrowhead=2, ax=ax, ay=ay,
                       font=dict(color=GOLD if "Specific" in txt else GRAY, size=10))

fig.update_layout(
    title=dict(
        text=(
            "<b>Hyperbolic Embedding Space — Poincaré Disk</b>"
            "<br><sup>exp-map projection of 16-D FinBERT embeddings · "
            "Boundary distance encodes event hierarchy · Hover → headlines</sup>"
        ),
        font=dict(color=TXT, size=16),
    ),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
               scaleanchor="y", range=[-1.15, 1.15]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
               range=[-1.15, 1.15]),
    height=580,
    **_dark(margin=dict(l=20, r=20, t=100, b=20)),
)
fig.show()

---
## 4 · Chebyshev Spectral Convolution

After graph construction, node features are aggregated via an order-$K$ Chebyshev expansion of the graph signal:

$$\mathbf{z} = \sum_{k=0}^{K-1} \theta_k \, T_k\!\left(\tilde{L}\right)\mathbf{x}, \qquad \tilde{L} = \tfrac{2}{\lambda_{\max}}L_{\text{norm}} - I$$

- $T_k$ are Chebyshev polynomials on $[-1,1]$, computed by the recursion $T_k(x) = 2x\,T_{k-1}(x) - T_{k-2}(x)$
- $\lambda_{\max}$ = largest eigenvalue of the normalised Laplacian
- $\theta_k$ are **learnable filter coefficients**
- A residual skip connection adds the un-convolved input: $\mathbf{z} \mathrel{+}= \mathbf{x}$

This is a *spectral* low-pass filter: low-frequency (smooth) graph signals are preserved, high-frequency (noisy) ones are attenuated.

In [ ]:
import scipy.linalg as sla

# ── Graph Laplacian from the 60-node news graph ───────────────────────────────
D_mat   = np.diag(adj_knn.sum(axis=1))
D_inv_s = np.diag(1.0 / (np.sqrt(D_mat.diagonal() + 1e-6)))
L_norm  = D_inv_s @ (D_mat - adj_knn) @ D_inv_s   # symmetric normalised

eigvals = np.sort(sla.eigvalsh(L_norm))
lam_max = float(eigvals[-1])
print(f"λ_max = {lam_max:.4f}   (all λ ∈ [0, 2] for normalised Laplacian)")

# ── Chebyshev polynomials over [-1, 1] ────────────────────────────────────────
K     = 5
x_ch  = np.linspace(-1, 1, 600)
T     = [np.ones_like(x_ch), x_ch.copy()]
for _ in range(2, K):
    T.append(2 * x_ch * T[-1] - T[-2])

cheb_colors = [GOLD, GREEN, CYAN, PURP, RED]
cheb_labels = [f"T₀  (DC / identity)", f"T₁  (linear)",
               f"T₂  (quadratic)", f"T₃  (cubic)", f"T₄  (quartic)"]

# ── Figure ───────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Chebyshev Basis Functions  T_k(λ̃)", "Graph Laplacian Eigenvalue Spectrum"],
    horizontal_spacing=0.12,
)

# Left — polynomial curves
for k, (Tk, col, lbl) in enumerate(zip(T, cheb_colors, cheb_labels)):
    fig.add_trace(go.Scatter(
        x=x_ch, y=Tk, mode="lines", name=lbl,
        line=dict(color=col, width=2.4),
        hovertemplate=f"λ̃=%{{x:.3f}}<br>T{k}=%{{y:.3f}}<extra>{lbl}</extra>",
    ), row=1, col=1)

fig.add_hline(y=0, line_dash="dot", line_color=GRAY, line_width=0.8, row=1, col=1)

# Right — eigenvalue histogram
fig.add_trace(go.Histogram(
    x=eigvals, nbinsx=20,
    marker_color=GOLD, opacity=0.82,
    name="Eigenvalues",
    hovertemplate="λ = %{x:.3f}<br>Count = %{y}<extra></extra>",
), row=1, col=2)

fig.add_vline(x=lam_max, line_dash="dash", line_color=RED, line_width=1.8, row=1, col=2)
fig.add_annotation(
    x=lam_max, y=0, yref="paper", text=f"λ_max = {lam_max:.2f}",
    arrowcolor=RED, arrowhead=2, ay=-45, font=dict(color=RED, size=11), row=1, col=2,
)

# Shade "low-pass" region (left half of spectrum)
fig.add_vrect(x0=-1, x1=0, fillcolor="rgba(46,204,113,0.07)",
              line_width=0, annotation_text="Low-freq (preserved)",
              annotation_font_color=GREEN, annotation_position="top left",
              row=1, col=1)
fig.add_vrect(x0=0, x1=1, fillcolor="rgba(231,76,60,0.07)",
              line_width=0, annotation_text="High-freq (attenuated)",
              annotation_font_color=RED, annotation_position="top right",
              row=1, col=1)

fig.update_layout(
    height=460, showlegend=True,
    legend=dict(bgcolor="rgba(0,0,0,0)", x=0.01, y=0.99, font=dict(color=TXT)),
    **_dark(title=dict(
        text="<b>Chebyshev Spectral Convolution</b>  ·  K = 5 polynomial filter over the news graph",
        font=dict(color=TXT, size=17))),
)
fig.update_xaxes(title_text="Rescaled eigenvalue λ̃ ∈ [−1, 1]", gridcolor=GRID,
                 zerolinecolor=GRID, tickfont=dict(color=GRAY), row=1, col=1)
fig.update_yaxes(title_text="T_k(λ̃)", gridcolor=GRID, zerolinecolor=GRID,
                 range=[-1.25, 1.25], tickfont=dict(color=GRAY), row=1, col=1)
fig.update_xaxes(title_text="λ (graph Laplacian eigenvalue)", gridcolor=GRID,
                 zerolinecolor=GRID, tickfont=dict(color=GRAY), row=1, col=2)
fig.update_yaxes(title_text="Count", gridcolor=GRID, zerolinecolor=GRID,
                 tickfont=dict(color=GRAY), row=1, col=2)
fig.show()

---
## 5 · Hybrid Hyperbolic Attention

The core novelty of GSHA is a **per-head hybrid attention score**:

$$\text{score}_h(q, k) = \underbrace{\frac{\alpha_h}{\sqrt{d_h}} \langle q,\, k\rangle}_{\text{Euclidean}} - \underbrace{\beta_h \cdot c \cdot d_c^2\!\bigl(\exp_{\mathbf{0}}(q),\, \exp_{\mathbf{0}}(k)\bigr)}_{\text{Poincaré penalty}}$$

- $\alpha_h, \beta_h \ge 0$ via Softplus — **learnable per-head balance**
- $c \ge 0.1$ — **learnable curvature**
- $d_c(\cdot,\cdot)$ — geodesic distance on the Poincaré ball
- A **strict causal mask** zeros out all future tokens (upper triangle → $-\infty$)

The heatmap below shows softmax-normalised attention weights for a 30-day window.

In [ ]:
# ── Compute hybrid attention on a 30-day sample ───────────────────────────────
WIN = 30
win_dates = viz_dates[-WIN:]
E_win = torch.tensor(
    np.stack([embeddings[d].numpy() for d in win_dates]), dtype=torch.float32
)  # (30, 16)

# Typical trained hyper-parameters
alpha_h, beta_h, c_hyp = 0.62, 0.38, 0.50
d_h = E_win.shape[-1]

def exp_map0(v, c=1.0, eps=1e-7):
    nv = v.norm(dim=-1, keepdim=True).clamp(min=eps)
    return v.tanh() if v.shape[-1] == 1 else (torch.tanh(c**0.5 * nv) * v / (c**0.5 * nv))

def poincare_dsq(x, y, c=1.0, eps=1e-7):
    # Squared geodesic distance on Poincaré ball (vectorised, shape (N,N))
    # Möbius addition -x ⊕ y
    x2 = (x * x).sum(-1, keepdim=True).clamp(min=0)
    y2 = (y * y).sum(-1, keepdim=True).clamp(min=0)
    xy = (x * y).sum(-1, keepdim=True)
    num   = ((1 + 2*c*xy + c*y2) * x - (1 - c*x2) * y)
    denom = (1 + 2*c*xy + c**2 * x2 * y2).clamp(min=eps)
    mob   = -num / denom  # -x ⊕ y
    mob_n = mob.norm(dim=-1).clamp(min=0, max=1 - eps)
    return (2 / c**0.5 * torch.arctanh(c**0.5 * mob_n)) ** 2

with torch.no_grad():
    # Euclidean scores
    q   = E_win / d_h**0.5
    eucl = torch.mm(q, E_win.T)                  # (30, 30)

    # Poincaré scores
    qh = exp_map0(E_win, c_hyp)                  # map to disk
    # broadcast (N, 1, D) vs (1, N, D)
    qh_i = qh.unsqueeze(1).expand(WIN, WIN, d_h)
    qh_j = qh.unsqueeze(0).expand(WIN, WIN, d_h)
    dsq  = poincare_dsq(qh_i.reshape(-1, d_h),
                        qh_j.reshape(-1, d_h), c_hyp).reshape(WIN, WIN)

    hybrid = alpha_h * eucl - beta_h * c_hyp * dsq

    # Causal mask
    mask = torch.triu(torch.ones(WIN, WIN, dtype=torch.bool), diagonal=1)
    hybrid.masked_fill_(mask, float("-inf"))

    attn = torch.softmax(hybrid, dim=-1).numpy()

short_d = [d[5:] for d in win_dates]   # "MM-DD"

# ── Heatmap ───────────────────────────────────────────────────────────────────
fig = go.Figure(go.Heatmap(
    z=attn, x=short_d, y=short_d,
    colorscale=[
        [0.00, BG],
        [0.25, "#102040"],
        [0.55, DBLUE],
        [0.80, GOLD],
        [1.00, "#FFE066"],
    ],
    hovertemplate=(
        "Query: <b>%{y}</b><br>"
        "Key  : <b>%{x}</b><br>"
        "Attn : <b>%{z:.4f}</b><extra></extra>"
    ),
    colorbar=dict(
        title=dict(text="Attention Weight", font=dict(color=TXT)),
        tickfont=dict(color=TXT), thickness=12, len=0.7,
    ),
))

# Diagonal guide
fig.add_shape(type="line", x0=-0.5, y0=-0.5, x1=WIN-0.5, y1=WIN-0.5,
              line=dict(color=GREEN, width=0.8, dash="dot"))
# Causal boundary label
fig.add_annotation(
    x=WIN*0.72, y=WIN*0.28,
    text="▲ Causal mask<br>(upper triangle = −∞)",
    font=dict(color=GRAY, size=10), showarrow=False,
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Hybrid Hyperbolic Attention — Causal Weight Matrix</b>"
            f"<br><sup>Window: {win_dates[0]} → {win_dates[-1]} · "
            f"α={alpha_h:.2f} · β={beta_h:.2f} · curvature c={c_hyp}</sup>"
        ),
        font=dict(color=TXT, size=16),
    ),
    xaxis=dict(title="Key (date MM-DD)", tickangle=-45,
               tickfont=dict(size=9, color=GRAY), gridcolor="rgba(0,0,0,0)"),
    yaxis=dict(title="Query (date MM-DD)",
               tickfont=dict(size=9, color=GRAY), gridcolor="rgba(0,0,0,0)"),
    height=560,
    **_dark(margin=dict(l=80, r=40, t=100, b=90)),
)
fig.show()

---
## 6 · GSHA Architecture — Information Flow

The complete GSHA forward pass in six stages:

```
 Price OHLCV+indicators  ──► Price BiLSTM ──► Causal Self-Attention ──┐
                                                                        ├──► Cross-Modal Gate ──► MLP Head ──► ↑/↓
 News Embedding (16-D)   ──► Graph Builder ──► Chebyshev Conv ──► Causal Self-Attention ──► Hybrid Hyperbolic Attn ──┘
 News Mask (0/1) ────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ── GSHA Architecture Sankey ──────────────────────────────────────────────────
labels = [
    # 0-2  inputs
    "Price OHLCV<br>& Indicators",
    "News Embedding<br>(16-D FinBERT + PCA)",
    "News Mask  m_t ∈ {0,1}",
    # 3-8  processing
    "Price BiLSTM<br>(2-layer bidirectional)",
    "Dynamic Graph<br>Builder  (α, β, γ learnable)",
    "Chebyshev SpectralConv<br>(K=5, residual skip)",
    "Causal Self-Attention<br>on Price States",
    "Causal Self-Attention<br>on News Features",
    "Hybrid Hyperbolic<br>Attention  (α_h, β_h, c)",
    # 9-11 fusion & output
    "Cross-Modal<br>Confidence Gate",
    "MLP Head<br>(LayerNorm → GELU → Linear)",
    "Direction Prediction<br>↑ Up  /  ↓ Down",
]

node_colors = [
    "#1A6B9A", "#B8730A", "#4A5568",   # inputs
    "#2E86AB", "#D4A017", "#E67E22",    # processing 1
    "#27AE60", "#27AE60", "#C0392B",    # processing 2
    "#8E44AD",                           # fusion
    "#2C3E50",                           # head
    "#C0392B",                           # output
]

sources = [0, 1,  2,  1,  4,  3, 5,  6, 7,  8,  9]
targets = [3, 4,  4,  5,  5,  6, 7,  8, 8,  9, 10]
values  = [6, 5,  2,  5,  5,  6, 5,  6, 5,  6,  6]
link_col= [
    "rgba(46,134,171,0.42)",  "rgba(212,160,23,0.42)",  "rgba(74,85,104,0.30)",
    "rgba(212,160,23,0.42)",  "rgba(230,126,34,0.42)",  "rgba(46,204,113,0.42)",
    "rgba(46,204,113,0.42)",  "rgba(192,57,43,0.42)",   "rgba(192,57,43,0.42)",
    "rgba(142,68,173,0.42)",  "rgba(44,62,80,0.55)",
]

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=22, thickness=26,
        line=dict(color="rgba(255,255,255,0.0)", width=0),
        label=labels, color=node_colors,
        hovertemplate="<b>%{label}</b><br>Total flow: %{value}<extra></extra>",
    ),
    link=dict(
        source=sources, target=targets, value=values, color=link_col,
        hovertemplate="%{source.label}<br>→ %{target.label}<br>Flow: %{value}<extra></extra>",
    ),
))
fig.update_layout(
    title=dict(
        text=(
            "<b>GSHA — End-to-End Architecture</b>"
            "<br><sup>Graph-Spectral Hyperbolic Attention for Wheat Futures Direction Prediction</sup>"
        ),
        font=dict(color=TXT, size=18),
    ),
    height=540,
    paper_bgcolor=PBGC,
    font=dict(color=TXT, size=11),
    margin=dict(l=20, r=20, t=110, b=20),
)
fig.show()

---
## Summary

| Building Block | Role in GSHA | Visualized Above |
|---|---|---|
| **Dynamic Graph Builder** | Connects news days via semantic + temporal similarity | §2 — Interactive graph |
| **Poincaré Ball ops** | Encodes event hierarchy into curved geometry | §3 — Poincaré disk |
| **Chebyshev SpectralConv** | Low-pass aggregation of graph-neighbour signals | §4 — Spectral filter |
| **Causal Self-Attention** | Temporal causality on both price & news streams | §5 (masking) |
| **Hybrid Hyperbolic Attn** | Euclidean + Poincaré distance attention blend | §5 — Attention map |
| **Cross-Modal Gate** | Learned confidence weighting of price vs. news | §6 — Sankey |

> Full training experiments and ablation results: see `GSHA_Research.ipynb`